# 14 — Бюджет латентности 40 секунд

> **Мягкое ограничение** от ментора: live-demo не должно «зависать»
> больше 40 секунд на один прогон цикла.
>
> **Решение (ADR-0008/ADR-0009):** budget cap в state + graceful
> degradation, если превышен — finalize с лучшим имеющимся SQL.

## Что покажем

1. Симулируем времена узлов через `time.sleep` (масштаб ×100 уменьшен
   чтобы ноутбук не тормозил: «5 сек» → 50 мс).
2. LoopState с tracker'ом времени и токенов.
3. **Sample run** (2 итерации) → укладываемся.
4. **Stress run** (5 итераций) → превышаем → degraded result.
5. Гистограмма p50/p95/p99 на 100 синтетических прогонах.


## 🧒 Аналогия для ребёнка

Ты копишь карманные деньги — у тебя есть **40 рублей в день**.
Хочешь купить мороженое (10₽), пирожок (15₽), сок (10₽), жвачку (10₽).
В сумме — 45₽, не хватает.

- **Без бюджета:** покупаешь всё подряд, не считая. К концу дня
  узнаёшь что вышел в минус — у мамы выпрашиваешь.
- **С бюджетом:** считаешь по ходу. Купил мороженое+пирожок+сок=35₽.
  Подходишь к жвачке — видишь, остаётся 5₽ — отказываешься от неё.
  Это **graceful degradation**: жвачку не получил, но и в минус не ушёл.

В нашем цикле «время» = «бюджет». Если итерация 3 уже потратила 30 сек —
можно ли запускать reflector + iter 4? Считаем.


## 1. Симуляция времён узлов


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Времена узлов цикла (масштабировано ×100: «5 сек» → 50 мс).
# @details
#   В Colab эти задержки реальны — ноутбук будет действительно ждать.
#   Используем небольшие числа, чтобы общее время прогона ноутбука < 30 сек.
SCALE = 0.01  # «секунды» → миллисекунды

NODE_DURATIONS = {
    "schema_link":      0.5,
    "generator":        7.0,
    "auditor_phase1":   1.0,
    "auditor_phase2":   4.0,
    "reflector":        2.0,
    "finalize":         0.3,
}


##
# @brief Сценарий одного запуска: список узлов в порядке.
def make_pipeline(n_iterations):
    p = ["schema_link"]
    for _ in range(n_iterations):
        p += ["generator", "auditor_phase1", "auditor_phase2"]
    # Если итераций > 1, между ними был reflector (max_iter - 1 раз)
    if n_iterations > 1:
        # вставляем reflector между парами (упрощённо — не точные позиции)
        pass
    p.append("finalize")
    return p


print("Узлы и их «сек»:")
for n, d in NODE_DURATIONS.items():
    print(f"  {n:18s}  {d:>5.1f} sec")
print(f"\nМасштаб времени в симуляции: SCALE = {SCALE} (1 «сек» = {SCALE*1000:.0f} ms)")


## 2. State с budget cap


In [ ]:
from dataclasses import dataclass


@dataclass
class BudgetState:
    iteration: int = 0
    total_seconds: float = 0.0       # «секунды» (виртуальные)
    total_tokens: int = 0
    budget_seconds: float = 45.0
    budget_tokens: int = 80_000
    budget_exhausted: bool = False
    final_sql: str = ""

    def add(self, node, sec, tokens):
        self.total_seconds += sec
        self.total_tokens += tokens
        if self.total_seconds > self.budget_seconds or self.total_tokens > self.budget_tokens:
            self.budget_exhausted = True


def run_one_iter(state, with_reflector):
    """@brief Прогон одной итерации цикла."""
    state.iteration += 1
    if with_reflector:
        time.sleep(NODE_DURATIONS["reflector"] * SCALE)
        state.add("reflector", NODE_DURATIONS["reflector"], 200)
        if state.budget_exhausted: return
    for node in ("generator", "auditor_phase1", "auditor_phase2"):
        time.sleep(NODE_DURATIONS[node] * SCALE)
        # Имитация: generator/auditor тратят токены
        tokens = {"generator": 5000, "auditor_phase1": 0, "auditor_phase2": 3000}[node]
        state.add(node, NODE_DURATIONS[node], tokens)
        if state.budget_exhausted:
            return


def simulate_loop(n_iter_required):
    """@brief Полный прогон цикла."""
    state = BudgetState()
    time.sleep(NODE_DURATIONS["schema_link"] * SCALE)
    state.add("schema_link", NODE_DURATIONS["schema_link"], 0)
    for i in range(n_iter_required):
        run_one_iter(state, with_reflector=(i > 0))
        if state.budget_exhausted:
            state.final_sql = "SQL_from_last_valid_iter (graceful)"
            return state
    time.sleep(NODE_DURATIONS["finalize"] * SCALE)
    state.add("finalize", NODE_DURATIONS["finalize"], 0)
    state.final_sql = "SQL_approved"
    return state


## 3. Sample run — 2 итерации укладываются в бюджет


In [ ]:
section("Sample run: 2 итерации (типичный случай)")
import time as _t
t0 = _t.time()
s = simulate_loop(2)
real_elapsed = _t.time() - t0

print(f"  iterations:        {s.iteration}")
print(f"  итого «секунд»:    {s.total_seconds:.1f} / {s.budget_seconds}")
print(f"  итого «токенов»:   {s.total_tokens} / {s.budget_tokens}")
print(f"  budget_exhausted:  {s.budget_exhausted}")
print(f"  final_sql:         {s.final_sql}")
print(f"  (реально в Colab прошло: {real_elapsed*1000:.0f} ms)")


## 4. Stress run — 5 итераций, бюджет лопается


In [ ]:
section("Stress run: 5 итераций (worst case)")
t0 = _t.time()
s = simulate_loop(5)
real_elapsed = _t.time() - t0

print(f"  iterations:        {s.iteration}")
print(f"  итого «секунд»:    {s.total_seconds:.1f} / {s.budget_seconds}")
print(f"  budget_exhausted:  {s.budget_exhausted}")
print(f"  final_sql:         {s.final_sql}  ← graceful degradation")
print(f"  (реально в Colab прошло: {real_elapsed*1000:.0f} ms)")


## 5. Распределение латентности по 100 прогонам


In [ ]:
import random


def simulate_with_jitter():
    """@brief Прогон с реальным распределением: 80% — 2 итер, 15% — 3, 5% — 5."""
    rng = random.random()
    if rng < 0.80:
        n = 2
    elif rng < 0.95:
        n = 3
    else:
        n = 5
    # Делаем без time.sleep — только считаем
    state = BudgetState()
    state.add("schema_link", NODE_DURATIONS["schema_link"], 0)
    for i in range(n):
        if i > 0:
            state.add("reflector", NODE_DURATIONS["reflector"] + random.uniform(-0.5, 0.5), 200)
            if state.budget_exhausted: break
        for node in ("generator", "auditor_phase1", "auditor_phase2"):
            d = NODE_DURATIONS[node] * random.uniform(0.8, 1.2)
            state.add(node, d, 3000)
            if state.budget_exhausted: break
        if state.budget_exhausted: break
    if not state.budget_exhausted:
        state.add("finalize", NODE_DURATIONS["finalize"], 0)
    return state


latencies = []
exhausted = 0
for _ in range(100):
    s = simulate_with_jitter()
    latencies.append(s.total_seconds)
    if s.budget_exhausted:
        exhausted += 1

latencies.sort()
p50 = latencies[50]
p95 = latencies[95]
p99 = latencies[99]

section("Распределение латентности (100 прогонов)")
print(f"  p50 (median):           {p50:>5.1f} sec")
print(f"  p95:                    {p95:>5.1f} sec")
print(f"  p99:                    {p99:>5.1f} sec")
print(f"  превысили бюджет:       {exhausted}/100  ({exhausted}%)")
print()
print("  Цель ADR-0008 §5:")
print("    p50 ≤ 25 sec  →  выполнено" if p50 <= 25 else "    p50 ≤ 25 sec  →  ❌")
print(f"    p95 ≤ 40 sec  →  {'выполнено' if p95 <= 40 else '❌'}")
print(f"    exhausted ≤ 5%  →  {'выполнено' if exhausted <= 5 else '❌'}")

# Простая ASCII-гистограмма
section("ASCII-гистограмма (по 5-сек бакетам)")
buckets = {}
for l in latencies:
    b = int(l // 5) * 5
    buckets[b] = buckets.get(b, 0) + 1
for b in sorted(buckets):
    bar = "█" * buckets[b]
    print(f"  {b:>3}-{b+5:<3} sec  {bar} ({buckets[b]})")


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/05-latency-budget/README.md](../../problems/engineering/05-latency-budget/README.md)
- **Варианты решения + почему так:** [problems/engineering/05-latency-budget/solutions.md](../../problems/engineering/05-latency-budget/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
